# TAPER — last index, list edges, SQL changes

Closed-book. This is the gate before the next timed paper. A part is done when its test has been executed and the print is still in this notebook.

**Rules**
- Do not call `sorted` or `list.sort`.
- The empty case, the one-item case, and the last position are part of the answer.
- Checkpoints are at the bottom. Answers are only in `_archive/TAPER - answers.ipynb`.
- Copy the identifiers below. Do not reuse names from LINK, HCI, or the chains.


---
## T1 — insertion and bubble, including the last element

`batch` is the starting list. It is in reverse order, and `40` appears twice.

`InsertionSort(records)` returns a new list, sorted by score ascending. When two scores are equal, the record that appears earlier in `records` stays earlier. The input list must be unchanged.

`BubbleSort(records)` returns a new list, sorted by score descending, with the same rule for equal scores. The input list must be unchanged.

Both functions must return `[]` for an empty list and a one-item list unchanged.

**Test you must run and leave visible:**
- `print(InsertionSort(batch))`
- `print(BubbleSort(batch))`
- `print(InsertionSort([]))`
- `print(BubbleSort([("Z", 1)]))`
- `print(batch)` after both sorts, so the original order is still there


In [37]:
# T1 — YOUR ATTEMPT
# identifiers: batch, InsertionSort, BubbleSort

batch = [("E", 10), ("D", 40), ("C", 20), ("B", 40), ("A", 5)]

def InsertionSort(records):
    tosort = [_ for _ in records]
    for i in range(1,len(tosort)):
        current = tosort[i]
        j = i - 1
        while j >= 0 and current[1] < tosort[j][1]: #data seniority 
            tosort[j+1] = tosort[j]
            j -= 1
        tosort[j+1] = current
    return tosort

def BubbleSort(records):
    tosort = [_ for _ in records]
    length = len(records)
    swapped = True
    while swapped:
        swapped = False
        for n in range(length-1):
            if tosort[n][1] < tosort[n+1][1]:
                temp = tosort[n]
                tosort[n] = tosort[n+1]
                tosort[n+1] = temp
                swapped = True
    return tosort

print(InsertionSort(batch))
batch = [("E", 10), ("D", 40), ("C", 20), ("B", 40), ("A", 5)]
print(BubbleSort(batch))

[('A', 5), ('E', 10), ('C', 20), ('D', 40), ('B', 40)]
[('D', 40), ('B', 40), ('C', 20), ('E', 10), ('A', 5)]


---
## T2 — linked list edges, before the middle case

`Slip(num)` stores a private `num` and a private next slip. Provide `getNum`, `getNext`, `setNext`.

`Rack` starts empty.

`IsEmpty()` is `True` only when the rack has no slips.

`Add(num)` inserts in ascending order:

- empty rack: the new slip becomes first
- smaller than the first: it becomes the new first, and the old first follows it
- larger than every slip: it becomes the last slip
- duplicate `num`: print `Duplicate` and insert nothing
- otherwise: insert between two slips

`Remove(num)` returns `True` when a slip was removed and `False` when it was not:

- empty rack
- the first slip
- the last slip
- a number that is not in the rack

`Items()` walks with `getNext` and returns a list of numbers.

**Test you must run, in this order:** `Add` 30, 10, 40, 20, then 10 again. `Remove` 10, then 40, then 99. Print `Items()` after the adds and after each remove. Print `IsEmpty()` on a new empty rack.


In [39]:
# T2 — YOUR ATTEMPT
# identifiers: Slip, Rack, bay

bay = None

class Slip:
    def __init__(self, num = None):
        self.__num = num
        self.__next = None
        
    def getNum(self):
        return self.__num
    
    def getNext(self):
        return self.__next
    
    def setNext(self, Nxt):
        self.__next = Nxt
        
class Rack:
    def __init__(self):
        self.__head = None
        
    def IsEmpty(self):
        return self.__head is None
    
    def Add(self, num):
        new = Slip(num)
        #empty case
        if self.IsEmpty(): 
            self.__head = new
            return True
        #head case
        elif num < self.__head.getNum():
            new.setNext(self.__head)
            self.__head = new
            return True
        #middle and end cases
        previous = None
        current = self.__head
        while current is not None and current.getNum() <= num:
            if current.getNum() == num:
                print('duplicate')
                return False
            previous = current
            current = current.getNext()
        
        previous.setNext(new)
        new.setNext(current)
        return True
        
    def Remove(self, num):
        #empty case
        if self.IsEmpty():
            print("empty rack")
            return False
        #head case
        elif self.__head.getNum() == num:
            self.__head = self.__head.getNext()
            return True
        #middle case
        previous = None
        current = self.__head
        while current is not None and current.getNum() != num:
            previous = current
            current = current.getNext()
        #end case - reached end without finding anything = num
        if current is None:
            print("not in rack")
            return False
        previous.setNext(current.getNext())
        return True
    
    def Items(self):
        output = []
        current = self.__head
        while current is not None:
            output.append(current.getNum())
            current = current.getNext()
        return output
    
ni = Rack()
ni.Add(30)
print(ni.Items())
ni.Add(10)
print(ni.Items())
ni.Add(40)
print(ni.Items())
ni.Add(20)
print(ni.Items())
ni.Add(10)
print(ni.Items())
ni.Remove(10)
print(ni.Items())
ni.Remove(40)
print(ni.Items())
ni.Remove(99)
print(ni.Items())

[30]
[10, 30]
[10, 30, 40]
[10, 20, 30, 40]
duplicate
[10, 20, 30, 40]
[20, 30, 40]
[20, 30]
not in rack
[20, 30]


---
## T3 — an UPDATE and a DELETE that a second connection can see

Use the file `taper_jobs.db`. Table:

`Job(job_id INTEGER PRIMARY KEY, status TEXT)`

Seed job ids `1, 2, 3, 4`, each with status `open`.

`MarkDone(conn, job_id)` sets that one job to `done` and commits. Every other job stays `open`.

`Cancel(conn, first_id, second_id)` deletes those two job ids and commits. Every other job stays. The condition must name the column for each placeholder.

`Remaining(conn)` returns the surviving job ids in ascending order.

**Test you must run:** seed, `MarkDone` on `2`, print every `(job_id, status)`, `Cancel` 1 and 3, print `Remaining`.


In [25]:
# T3 — YOUR ATTEMPT
# identifiers: MarkDone, Cancel, Remaining
import sqlite3

def get_db():
    conn = sqlite3.connect('taper_jobs.db')
    conn.row_factory = sqlite3.Row
    return conn

conn = get_db()
cur = conn.cursor()
cur.execute("""
DROP TABLE IF EXISTS Job
""")
cur.execute("""
CREATE TABLE Job(job_id INTEGER PRIMARY KEY, status TEXT)
""")
for n in range(4):
    cur.execute("""
    INSERT OR IGNORE INTO Job(job_id, status)
    VALUES(?,?)
    """,(n,'open'))
conn.commit()

def MarkDone(conn, job_id):
    cur = conn.cursor()
    cur.execute("""
    UPDATE Job SET
    status = 'done'
    WHERE job_id = ?
    """, (job_id,))
    conn.commit()
    return True

def Cancel(conn, first_id, second_id):
    cur = conn.cursor()
    cur.execute("""
    DELETE FROM Job
    WHERE job_id = ? OR job_id = ?
    """, (first_id, second_id))
    conn.commit()
    return True

MarkDone(get_db(),1)
Cancel(get_db(), 2,3)
conn.close()

---
# Checkpoints

Run after the attempts. A FAIL means that part has no evidence yet. The SQL checkpoint removes `taper_jobs.db` when it finishes.


In [40]:
# CHECKPOINTS — TAPER
import os
import sqlite3

PARTS = ("T1", "T2", "T3")


def _check_t1():
    for fn in (InsertionSort, BubbleSort):
        src = fn.__code__.co_names
        if "sorted" in src or "sort" in src:
            return False, "sorted or list.sort is not allowed"
    original = [("E", 10), ("D", 40), ("C", 20), ("B", 40), ("A", 5)]
    got_i = InsertionSort(original)
    if original != [("E", 10), ("D", 40), ("C", 20), ("B", 40), ("A", 5)]:
        return False, "InsertionSort changed its input"
    if got_i != [("A", 5), ("E", 10), ("C", 20), ("D", 40), ("B", 40)]:
        return False, f"InsertionSort is {got_i}"
    got_b = BubbleSort(original)
    if got_b != [("D", 40), ("B", 40), ("C", 20), ("E", 10), ("A", 5)]:
        return False, f"BubbleSort is {got_b}"
    if InsertionSort([]) != [] or BubbleSort([]) != []:
        return False, "empty list must return []"
    if InsertionSort([("Z", 1)]) != [("Z", 1)] or BubbleSort([("Z", 1)]) != [("Z", 1)]:
        return False, "one-item list must come back unchanged"
    return True, "ok"


def _check_t2():
    rack = Rack()
    if rack.IsEmpty() is not True:
        return False, "empty rack must be empty"
    rack.Add(30)
    rack.Add(10)
    rack.Add(40)
    rack.Add(20)
    if rack.Items() != [10, 20, 30, 40]:
        return False, f"after adds, Items is {rack.Items()}"
    rack.Add(10)
    if rack.Items() != [10, 20, 30, 40]:
        return False, "duplicate was inserted"
    if rack.Remove(10) is not True or rack.Items() != [20, 30, 40]:
        return False, f"head remove left {rack.Items()}"
    if rack.Remove(40) is not True or rack.Items() != [20, 30]:
        return False, f"last remove left {rack.Items()}"
    if rack.Remove(99) is not False:
        return False, "missing number must return False"
    if Rack().Remove(1) is not False:
        return False, "remove on an empty rack must return False"
    if Rack().IsEmpty() is not True:
        return False, "a new rack must be empty"
    return True, "ok"


def _check_t3():
    path = "taper_jobs.db"
    if os.path.exists(path):
        os.remove(path)
    conn = sqlite3.connect(path)
    conn.execute("CREATE TABLE Job(job_id INTEGER PRIMARY KEY, status TEXT)")
    conn.executemany("INSERT INTO Job VALUES (?, ?)", [(1, "open"), (2, "open"), (3, "open"), (4, "open")])
    conn.commit()
    MarkDone(conn, 2)
    other = sqlite3.connect(path)
    rows = list(other.execute("SELECT job_id, status FROM Job ORDER BY job_id"))
    other.close()
    if rows != [(1, "open"), (2, "done"), (3, "open"), (4, "open")]:
        conn.close()
        return False, f"MarkDone visible rows are {rows}"
    Cancel(conn, 1, 3)
    check = sqlite3.connect(path)
    left = [r[0] for r in check.execute("SELECT job_id FROM Job ORDER BY job_id")]
    check.close()
    conn.close()
    os.remove(path)
    if left != [2, 4]:
        return False, f"Cancel left {left}"
    return True, "ok"


_CHECKS = {"T1": _check_t1, "T2": _check_t2, "T3": _check_t3}
for part in PARTS:
    try:
        ok, detail = _CHECKS[part]()
    except Exception as e:
        ok, detail = False, str(e)
    print(f"{part}: {'PASS' if ok else 'FAIL — ' + detail}")


T1: PASS
duplicate
not in rack
empty rack
T2: PASS
T3: PASS
